# PS-S3: Noise Accumulation in Trotter Circuits

### StepWise-Q: Optimal Trotter Step Prediction on Custom Quantum Topologies
**Qiskit Fall Fest 2026 | SRM University-AP | Track 3: Quantum Simulation**

> **One-sentence answer:** a tested rule, backed by theory and simulation on two processors we designed,
> that tells you the number of Trotter steps `n*` with the lowest total error, and how that number
> moves with evolution time, field strength, noise level and coupling topology.

---


## 0. Configuration

In [1]:
# ─── CONFIGURATION ────────────────────────────────────────────────────────────
# Set QUICK = True for a fast review run (reduced time grid, no layout search).
# Set QUICK = False for the full competition run.
QUICK = False

# Hard-wired constants (do not change — competition spec)
J          = 1.0
G_VALUES   = [0.5, 1.0, 1.5]
N_STEPS_REQUIRED  = [1, 2, 4]
N_STEPS_EXTENDED  = [1, 2, 3, 4, 6, 8]
T_GRID     = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0, 1.2, 1.4, 1.6, 1.8, 2.0] if QUICK else list(__import__('numpy').linspace(0, 2.0, 21))
T_GRID_RED = [0.5, 1.0, 1.5, 2.0]   # reduced grid for extensions
SHOTS      = 4096
SEED       = 42
PROC_IDS_A = ["A0", "A_star"]
PROC_IDS_B = ["B0", "B_star"]

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

print(f"QUICK mode: {QUICK}")
print(f"t_grid points: {len(T_GRID)}")


QUICK mode: False
t_grid points: 21


## 1. Imports and Setup

In [2]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path(".").resolve()))

from src.problem import (
    tfim_hamiltonian, initial_state, exact_trajectory,
    compute_observable, spin_to_qubit,
    validate_spin_to_qubit, validate_initial_observable, validate_norm_preserved,
)
from src.processors import (
    PROCESSOR_REGISTRY, load_processor, build_noise_model,
    graph_metrics, populate_all_graph_metrics, save_processor_json,
    validate_connectivity, validate_json_roundtrip,
)
from src.routing import (
    build_trotter_circuit, transpile_circuit, count_swaps,
    cx_identity_check, circuit_stats,
    simulate_density_matrix, simulate_shots, layout_search,
)
from src.metrics import (
    trotter_error_model_fit, predict_n_star, bootstrap_n_star_ci,
    recommend_steps, zne_extrapolate, readout_mitigation,
    noise_scale_sweep, run_validation,
)
from src.visualization import (
    fig1_coupling_maps, fig2_trajectory, fig3_u_curve, fig4_n_star_map,
    fig5_scaling_law, fig6_cx_depth, fig7_layout, fig8_mitigation,
    FIGURES_DIR,
)

# Populate graph metrics for all processors
populate_all_graph_metrics()

print("All imports successful.")
print(f"Processor registry: {list(PROCESSOR_REGISTRY.keys())}")


All imports successful.
Processor registry: ['A0', 'A1', 'A_star', 'B0', 'B_star']


## 2. Phase 1 — Hamiltonians and Exact Reference

We build the 4-spin TFIM Hamiltonian for all three field values and compute the exact
time evolution using `scipy.linalg.expm`. This serves as the ground truth for all error calculations.

**Model:** H = −J Σ Z_i Z_{i+1} − g Σ X_i, J=1.0, g ∈ {0.5, 1.0, 1.5}


In [3]:
import numpy as np

# Phase 1 gate checks
print("=== Phase 1 Gate Checks ===")
print(f"spin_to_qubit check:  {'PASS' if validate_spin_to_qubit() else 'FAIL'}")
print(f"Z0(0) check:          {'PASS' if validate_initial_observable() else 'FAIL'}")
print(f"norm preserved:       {'PASS' if validate_norm_preserved() else 'FAIL'}")
print()

# Build Hamiltonians
H_open = {g: tfim_hamiltonian(4, 1.0, g, "open") for g in G_VALUES}
H_ring = {g: tfim_hamiltonian(4, 1.0, g, "ring") for g in G_VALUES}

print("Hamiltonians built:")
for g in G_VALUES:
    print(f"  g={g}: n_terms={len(H_open[g])}")

# Initial states
sv_zeros = initial_state("zeros", 4)
sv_neel  = initial_state("neel", 4)

# Exact trajectories for g=1.0
t_arr = np.array(T_GRID)
observables = ["Z0", "mean_mag", "ZZ_nn", "energy"]

print("\nComputing exact trajectories (g=1.0)...")
exact_zeros = exact_trajectory(
    H_open[1.0], sv_zeros, t_arr, observables, save_prefix="open_zeros_g1"
)
exact_neel = exact_trajectory(
    H_open[1.0], sv_neel,  t_arr, observables, save_prefix="open_neel_g1"
)
print(f"  Z0(t=0)  zeros: {exact_zeros['Z0'][0]:.4f} (expected +1)")
print(f"  Z0(t=0)  neel:  {exact_neel['Z0'][0]:.4f}  (expected +1)")

# Compute for all g values
EXACT = {}   # {(boundary, g, init_state): {obs: array}}
for g in G_VALUES:
    for init_name, sv0 in [("zeros", sv_zeros), ("neel", sv_neel)]:
        for boundary, H_dict in [("open", H_open), ("ring", H_ring)]:
            key = (boundary, g, init_name)
            EXACT[key] = exact_trajectory(
                H_dict[g], sv0, t_arr, ["Z0"],
                save_prefix=f"{boundary}_{init_name}_g{g:.1f}",
            )

print(f"\nExact trajectories computed for {len(EXACT)} configurations.")


=== Phase 1 Gate Checks ===
spin_to_qubit check:  PASS
Z0(0) check:          PASS
norm preserved:       PASS

Hamiltonians built:
  g=0.5: n_terms=7
  g=1.0: n_terms=7
  g=1.5: n_terms=7

Computing exact trajectories (g=1.0)...
  Z0(t=0)  zeros: 1.0000 (expected +1)
  Z0(t=0)  neel:  1.0000  (expected +1)

Exact trajectories computed for 12 configurations.


## 3. Phase 2 — Processor Topologies

We benchmark five coupling maps: two reference topologies (A0, B0) and three custom designs (A1, A*, B*).
Our final submission uses **A*** (5-qubit) and **B*** (9-qubit).


In [4]:
# Phase 2 gate checks
print("=== Phase 2 Gate Checks ===")
conn = validate_connectivity()
for pid, ok in conn.items():
    print(f"  {pid} connectivity: {'PASS' if ok else 'FAIL'}")
print(f"  JSON round-trip: {'PASS' if validate_json_roundtrip() else 'FAIL'}")
print()

# Print graph metrics table
import pandas as pd
rows = []
for pid, proc in PROCESSOR_REGISTRY.items():
    m = proc["graph_metrics"]
    rows.append({
        "Processor": pid,
        "Qubits": proc["n_qubits"],
        "Edges": m["n_edges"],
        "Diameter": m["diameter"],
        "Avg degree": round(m["avg_degree"], 2),
        "Max degree": m["max_degree"],
        "Connected": m["is_connected"],
    })
df_metrics = pd.DataFrame(rows)
display(df_metrics)

# Save initial JSON files
save_processor_json("A_star", pathlib.Path("processors/processor_A.json"))
save_processor_json("B_star", pathlib.Path("processors/processor_B.json"))
print("Wrote processors/processor_A.json (A*) and processor_B.json (B*)")


=== Phase 2 Gate Checks ===
  A0 connectivity: PASS
  A1 connectivity: PASS
  A_star connectivity: PASS
  B0 connectivity: PASS
  B_star connectivity: PASS
  JSON round-trip: PASS



,Processor,Qubits,Edges,Diameter,Avg degree,Max degree,Connected
0,A0,5,4,4,1.60,2,True
1,A1,5,4,3,1.60,3,True
2,A_star,5,5,3,2.00,3,True
3,B0,9,12,4,2.67,5,True
4,B_star,9,10,6,2.22,3,True


Wrote processors/processor_A.json (A*) and processor_B.json (B*)


In [5]:
# Figure 1: Coupling maps
procs = [PROCESSOR_REGISTRY[p] for p in ("A0", "A1", "A_star", "B0", "B_star")]
fig1_coupling_maps(procs)
plt.show()
print(f"Fig 1 saved to {FIGURES_DIR / 'fig1_coupling_maps.png'}")


Fig 1 saved to /Users/pavan/Desktop/project/results/figures/fig1_coupling_maps.png


## 4. Phase 3 — Trotter Circuits and SWAP Counting

We build, transpile and measure the gate resources for each processor/Hamiltonian combination.
The key validation: open chain on A0, A*, B0, B* must require **zero SWAP gates**.


In [6]:
print("=== Phase 3 Gate Checks: SWAP validation ===")
from src.routing import validate_swap_counts
swap_checks = validate_swap_counts(g=1.0)
for name, ok in swap_checks.items():
    print(f"  {name}: {'PASS' if ok else 'FAIL'}")
print()

# Sample transpiled circuit stats
H_test = tfim_hamiltonian(4, 1.0, 1.0, "open")
stats_rows = []
for pid in ("A0", "A_star", "B0", "B_star"):
    proc = PROCESSOR_REGISTRY[pid]
    for n_steps in N_STEPS_REQUIRED:
        qc = build_trotter_circuit(H_test, 2.0, n_steps, order=1)
        qc_t = transpile_circuit(qc, proc)
        st = circuit_stats(qc_t)
        sw = count_swaps(qc, proc)
        stats_rows.append({
            "Processor": pid, "n": n_steps, "Depth": st["depth"],
            "CX": st["cx"], "SWAP (open)": sw,
        })

df_stats = pd.DataFrame(stats_rows)
display(df_stats)


=== Phase 3 Gate Checks: SWAP validation ===


  open_swap_zero_A0: PASS
  open_swap_zero_A_star: PASS
  open_swap_zero_B0: PASS
  open_swap_zero_B_star: PASS
  ring_swap_gt0_A0: PASS
  ring_swap_zero_Astar: PASS



,Processor,n,Depth,CX,SWAP (open)
0,A0,1,14,6,0
1,A0,2,25,12,0
2,A0,4,47,24,0
3,A_star,1,14,6,0
4,A_star,2,25,12,0
5,A_star,4,47,24,0
6,B0,1,14,6,0
7,B0,2,25,12,0
8,B0,4,47,24,0
9,B_star,1,14,6,0


## 5. Phase 4 — Required Run (R0)

Open chain, order-1 Trotter, n ∈ {1, 2, 4}, g ∈ {0.5, 1.0, 1.5}, both initial states,
processors A0, A*, B0, B*. This is the **safe submission baseline**.


In [7]:
import pathlib, csv
pathlib.Path("results/tables").mkdir(parents=True, exist_ok=True)

benchmark_rows = []

proc_ids_all = ["A0", "A_star", "B0", "B_star"]

for pid in proc_ids_all:
    proc = PROCESSOR_REGISTRY[pid]
    for g in G_VALUES:
        for init_name in ["zeros", "neel"]:
            sv0 = initial_state(init_name, 4)
            H_g = tfim_hamiltonian(4, 1.0, g, "open")
            
            # Exact at t=2.0
            exact_at_t2 = EXACT[("open", g, init_name)]["Z0"][-1]
            
            for n_steps in N_STEPS_REQUIRED:
                qc = build_trotter_circuit(H_g, 2.0, n_steps, order=1)
                qc_t = transpile_circuit(qc, proc)
                st  = circuit_stats(qc_t)
                sw  = count_swaps(qc, proc)
                
                # Noisy (density matrix)
                dm_res = simulate_density_matrix(qc_t, proc, ["Z0"])
                z0_noisy = dm_res["Z0"]
                
                # Noiseless
                from qiskit_aer import AerSimulator
                from qiskit.quantum_info import DensityMatrix as QiDM
                sim_nl = AerSimulator(method="density_matrix")
                qc_nl = qc_t.copy(); qc_nl.save_density_matrix()
                res_nl = sim_nl.run(qc_nl, shots=1, seed_simulator=42).result()
                dm_nl  = QiDM(res_nl.data(0)["density_matrix"])
                z0_noiseless = compute_observable(dm_nl, "Z0", 4)
                
                benchmark_rows.append({
                    "processor": pid, "n_qubits": proc["n_qubits"],
                    "boundary": "open", "order": 1, "g": g,
                    "init_state": init_name, "n_steps": n_steps,
                    "t": 2.0,
                    "Z0_exact": round(exact_at_t2, 6),
                    "Z0_noiseless": round(z0_noiseless, 6),
                    "Z0_noisy": round(z0_noisy, 6),
                    "error_noiseless": round(abs(exact_at_t2 - z0_noiseless), 6),
                    "error_noisy":     round(abs(exact_at_t2 - z0_noisy), 6),
                    "depth": st["depth"], "cx": st["cx"], "swap": sw,
                })
                print(f"  {pid} g={g} {init_name} n={n_steps}: "
                      f"Z0_ex={exact_at_t2:.4f} Z0_noisy={z0_noisy:.4f} SWAP={sw}")

df_bench = pd.DataFrame(benchmark_rows)
df_bench.to_csv("results/tables/benchmark_required.csv", index=False)
print(f"\nWrote results/tables/benchmark_required.csv ({len(df_bench)} rows)")
display(df_bench.head(10))


  A0 g=0.5 zeros n=1: Z0_ex=0.6409 Z0_noisy=-0.4029 SWAP=0
  A0 g=0.5 zeros n=2: Z0_ex=0.6409 Z0_noisy=0.5612 SWAP=0
  A0 g=0.5 zeros n=4: Z0_ex=0.6409 Z0_noisy=0.5774 SWAP=0
  A0 g=0.5 neel n=1: Z0_ex=0.6409 Z0_noisy=-0.4029 SWAP=0
  A0 g=0.5 neel n=2: Z0_ex=0.6409 Z0_noisy=0.5612 SWAP=0
  A0 g=0.5 neel n=4: Z0_ex=0.6409 Z0_noisy=0.5774 SWAP=0
  A0 g=1.0 zeros n=1: Z0_ex=0.0586 Z0_noisy=-0.6329 SWAP=0
  A0 g=1.0 zeros n=2: Z0_ex=0.0586 Z0_noisy=0.4949 SWAP=0
  A0 g=1.0 zeros n=4: Z0_ex=0.0586 Z0_noisy=0.0735 SWAP=0
  A0 g=1.0 neel n=1: Z0_ex=0.0586 Z0_noisy=-0.6329 SWAP=0


  A0 g=1.0 neel n=2: Z0_ex=0.0586 Z0_noisy=0.4949 SWAP=0
  A0 g=1.0 neel n=4: Z0_ex=0.0586 Z0_noisy=0.0735 SWAP=0
  A0 g=1.5 zeros n=1: Z0_ex=-0.0848 Z0_noisy=0.9297 SWAP=0
  A0 g=1.5 zeros n=2: Z0_ex=-0.0848 Z0_noisy=0.9456 SWAP=0
  A0 g=1.5 zeros n=4: Z0_ex=-0.0848 Z0_noisy=-0.0746 SWAP=0
  A0 g=1.5 neel n=1: Z0_ex=-0.0848 Z0_noisy=0.9297 SWAP=0
  A0 g=1.5 neel n=2: Z0_ex=-0.0848 Z0_noisy=0.9456 SWAP=0
  A0 g=1.5 neel n=4: Z0_ex=-0.0848 Z0_noisy=-0.0746 SWAP=0


  A_star g=0.5 zeros n=1: Z0_ex=0.6409 Z0_noisy=-0.4029 SWAP=0
  A_star g=0.5 zeros n=2: Z0_ex=0.6409 Z0_noisy=0.5612 SWAP=0
  A_star g=0.5 zeros n=4: Z0_ex=0.6409 Z0_noisy=0.5774 SWAP=0
  A_star g=0.5 neel n=1: Z0_ex=0.6409 Z0_noisy=-0.4029 SWAP=0
  A_star g=0.5 neel n=2: Z0_ex=0.6409 Z0_noisy=0.5612 SWAP=0
  A_star g=0.5 neel n=4: Z0_ex=0.6409 Z0_noisy=0.5774 SWAP=0
  A_star g=1.0 zeros n=1: Z0_ex=0.0586 Z0_noisy=-0.6329 SWAP=0
  A_star g=1.0 zeros n=2: Z0_ex=0.0586 Z0_noisy=0.4949 SWAP=0
  A_star g=1.0 zeros n=4: Z0_ex=0.0586 Z0_noisy=0.0735 SWAP=0
  A_star g=1.0 neel n=1: Z0_ex=0.0586 Z0_noisy=-0.6329 SWAP=0
  A_star g=1.0 neel n=2: Z0_ex=0.0586 Z0_noisy=0.4949 SWAP=0


  A_star g=1.0 neel n=4: Z0_ex=0.0586 Z0_noisy=0.0735 SWAP=0
  A_star g=1.5 zeros n=1: Z0_ex=-0.0848 Z0_noisy=0.9297 SWAP=0
  A_star g=1.5 zeros n=2: Z0_ex=-0.0848 Z0_noisy=0.9456 SWAP=0
  A_star g=1.5 zeros n=4: Z0_ex=-0.0848 Z0_noisy=-0.0746 SWAP=0
  A_star g=1.5 neel n=1: Z0_ex=-0.0848 Z0_noisy=0.9297 SWAP=0
  A_star g=1.5 neel n=2: Z0_ex=-0.0848 Z0_noisy=0.9456 SWAP=0
  A_star g=1.5 neel n=4: Z0_ex=-0.0848 Z0_noisy=-0.0746 SWAP=0


  B0 g=0.5 zeros n=1: Z0_ex=0.6409 Z0_noisy=-0.4029 SWAP=0
  B0 g=0.5 zeros n=2: Z0_ex=0.6409 Z0_noisy=0.5612 SWAP=0


...

  B_star g=0.5 neel n=2: Z0_ex=0.6409 Z0_noisy=0.5612 SWAP=0
  B_star g=0.5 neel n=4: Z0_ex=0.6409 Z0_noisy=0.5774 SWAP=0
  B_star g=1.0 zeros n=1: Z0_ex=0.0586 Z0_noisy=-0.6329 SWAP=0
  B_star g=1.0 zeros n=2: Z0_ex=0.0586 Z0_noisy=0.4949 SWAP=0


  B_star g=1.0 zeros n=4: Z0_ex=0.0586 Z0_noisy=0.0735 SWAP=0
  B_star g=1.0 neel n=1: Z0_ex=0.0586 Z0_noisy=-0.6329 SWAP=0
  B_star g=1.0 neel n=2: Z0_ex=0.0586 Z0_noisy=0.4949 SWAP=0


  B_star g=1.0 neel n=4: Z0_ex=0.0586 Z0_noisy=0.0735 SWAP=0
  B_star g=1.5 zeros n=1: Z0_ex=-0.0848 Z0_noisy=0.9297 SWAP=0
  B_star g=1.5 zeros n=2: Z0_ex=-0.0848 Z0_noisy=0.9456 SWAP=0


  B_star g=1.5 zeros n=4: Z0_ex=-0.0848 Z0_noisy=-0.0746 SWAP=0
  B_star g=1.5 neel n=1: Z0_ex=-0.0848 Z0_noisy=0.9297 SWAP=0
  B_star g=1.5 neel n=2: Z0_ex=-0.0848 Z0_noisy=0.9456 SWAP=0
  B_star g=1.5 neel n=4: Z0_ex=-0.0848 Z0_noisy=-0.0746 SWAP=0

Wrote results/tables/benchmark_required.csv (72 rows)


,processor,n_qubits,boundary,order,g,init_state,n_steps,t,Z0_exact,Z0_noiseless,Z0_noisy,error_noiseless,error_noisy,depth,cx,swap
0,A0,5,open,1,0.5,zeros,1,2.0,0.640867,-0.416147,-0.402949,1.057013,1.043815,14,6,0
1,A0,5,open,1,0.5,zeros,2,2.0,0.640867,0.586589,0.561196,0.054277,0.079671,25,12,0
2,A0,5,open,1,0.5,zeros,4,2.0,0.640867,0.623992,0.577426,0.016875,0.063440,47,24,0
3,A0,5,open,1,0.5,neel,1,2.0,0.640867,-0.416147,-0.402949,1.057013,1.043815,14,6,0
4,A0,5,open,1,0.5,neel,2,2.0,0.640867,0.586589,0.561196,0.054277,0.079671,25,12,0
5,A0,5,open,1,0.5,neel,4,2.0,0.640867,0.623992,0.577426,0.016875,0.063440,47,24,0
6,A0,5,open,1,1.0,zeros,1,2.0,0.058572,-0.653644,-0.632913,0.712215,0.691485,14,6,0
7,A0,5,open,1,1.0,zeros,2,2.0,0.058572,0.517257,0.494865,0.458686,0.436294,25,12,0
8,A0,5,open,1,1.0,zeros,4,2.0,0.058572,0.081400,0.073493,0.022828,0.014922,47,24,0
9,A0,5,open,1,1.0,neel,1,2.0,0.058572,-0.653644,-0.632913,0.712215,0.691485,14,6,0


In [8]:
# Phase 4 gate: zero-noise ≡ noiseless; noisy error >= noiseless error
print("=== Phase 4 Gate Checks ===")
all_ok = True
for _, row in df_bench.iterrows():
    # noisy error should be >= noiseless error (noise always adds error on average)
    if row["error_noisy"] < row["error_noiseless"] - 0.01:  # 0.01 tolerance for finite-sample effects
        print(f"  WARN: noisy < noiseless for {row['processor']} n={row['n_steps']} g={row['g']}")
        all_ok = False
print(f"  Noisy >= Noiseless check: {'PASS' if all_ok else 'WARN (see above)'}") 


=== Phase 4 Gate Checks ===
  WARN: noisy < noiseless for A0 n=1 g=0.5
  WARN: noisy < noiseless for A0 n=1 g=0.5
  WARN: noisy < noiseless for A0 n=1 g=1.0
  WARN: noisy < noiseless for A0 n=2 g=1.0
  WARN: noisy < noiseless for A0 n=1 g=1.0
  WARN: noisy < noiseless for A0 n=2 g=1.0
  WARN: noisy < noiseless for A0 n=1 g=1.5
  WARN: noisy < noiseless for A0 n=2 g=1.5
  WARN: noisy < noiseless for A0 n=1 g=1.5
  WARN: noisy < noiseless for A0 n=2 g=1.5
  WARN: noisy < noiseless for A_star n=1 g=0.5
  WARN: noisy < noiseless for A_star n=1 g=0.5
  WARN: noisy < noiseless for A_star n=1 g=1.0
  WARN: noisy < noiseless for A_star n=2 g=1.0
  WARN: noisy < noiseless for A_star n=1 g=1.0
  WARN: noisy < noiseless for A_star n=2 g=1.0
  WARN: noisy < noiseless for A_star n=1 g=1.5
  WARN: noisy < noiseless for A_star n=2 g=1.5
  WARN: noisy < noiseless for A_star n=1 g=1.5
  WARN: noisy < noiseless for A_star n=2 g=1.5
  WARN: noisy < noiseless for B0 n=1 g=0.5
  WARN: noisy < noiseless for

## 6. Phase 5 — Extensions

- **X1:** Finer n grid {1,2,3,4,6,8} for the error model fit
- **X2:** Ring Hamiltonian — topology comparison and SWAP overhead
- **X3:** Order-2 (Suzuki) Trotter
- Layout search skipped in QUICK mode.


In [9]:
n_steps_ext = N_STEPS_EXTENDED if not QUICK else N_STEPS_REQUIRED
ext_rows = []

# X1: finer n grid + X2: ring boundary
for pid in ["A_star", "B_star"]:  # Focus on our designed processors
    proc = PROCESSOR_REGISTRY[pid]
    for g in G_VALUES:
        for boundary in ["open", "ring"]:
            for order in [1, 2]:
                H_g = tfim_hamiltonian(4, 1.0, g, boundary)
                sv0 = initial_state("zeros", 4)
                exact_at_t2 = exact_trajectory(H_g, sv0, np.array([2.0]), ["Z0"])["Z0"][0]
                
                for n_steps in n_steps_ext:
                    qc = build_trotter_circuit(H_g, 2.0, n_steps, order=order)
                    qc_t = transpile_circuit(qc, proc)
                    st = circuit_stats(qc_t)
                    sw = count_swaps(qc, proc)
                    
                    dm_res = simulate_density_matrix(qc_t, proc, ["Z0"])
                    z0_noisy = dm_res["Z0"]
                    
                    from qiskit_aer import AerSimulator
                    from qiskit.quantum_info import DensityMatrix as QiDM
                    sim_nl = AerSimulator(method="density_matrix")
                    qc_nl = qc_t.copy(); qc_nl.save_density_matrix()
                    res_nl = sim_nl.run(qc_nl, shots=1, seed_simulator=42).result()
                    dm_nl = QiDM(res_nl.data(0)["density_matrix"])
                    z0_nl = compute_observable(dm_nl, "Z0", 4)
                    
                    ext_rows.append({
                        "processor": pid, "boundary": boundary, "order": order,
                        "g": g, "n_steps": n_steps, "t": 2.0,
                        "Z0_exact": round(exact_at_t2, 6),
                        "Z0_noiseless": round(z0_nl, 6),
                        "Z0_noisy": round(z0_noisy, 6),
                        "error_noiseless": round(abs(exact_at_t2 - z0_nl), 6),
                        "error_noisy": round(abs(exact_at_t2 - z0_noisy), 6),
                        "depth": st["depth"], "cx": st["cx"], "swap": sw,
                    })

df_ext = pd.DataFrame(ext_rows)
df_ext.to_csv("results/tables/benchmark_extended.csv", index=False)
print(f"Extended benchmark: {len(df_ext)} rows saved.")
display(df_ext.groupby(["processor","boundary","order","g"])[["error_noiseless","error_noisy","swap"]].mean().round(4))


Extended benchmark: 144 rows saved.


error_noiseless  error_noisy  swap
processor boundary order g                                      
A_star    open     1     0.5           0.1949       0.2354   0.0
                         1.0           0.2092       0.1986   0.0
                         1.5           0.3769       0.3706   0.0
                   2     0.5           0.1949       0.2435   0.0
                         1.0           0.2092       0.1973   0.0
                         1.5           0.3769       0.3719   0.0
          ring     1     0.5           0.3447       0.4357   0.0
                         1.0           0.4368       0.5001   0.0
                         1.5           0.3928       0.3753   0.0
                   2     0.5           0.3447       0.4724   0.0
                         1.0           0.4368       0.5275   0.0
                         1.5           0.3928       0.3691   0.0
B_star    open     1     0.5           0.1949       0.2354   0.0
                         1.0           0.2092       0.1986   0.0
                         1.5           0.3769       0.3706   0.0
                   2     0.5           0.1949       0.2435   0.0
                         1.0           0.2092       0.1973   0.0
                         1.5           0.3769       0.3719   0.0
          ring     1     0.5           0.3447       0.4357   0.0
                         1.0           0.4368       0.5001   0.0
                         1.5           0.3928       0.3753   0.0
                   2     0.5           0.3447       0.4724   0.0
                         1.0           0.4368       0.5275   0.0
                         1.5           0.3928       0.3691   0.0

## 7. Phase 6 — Error Model Fit and n* Estimation

We fit the two-parameter model to determine the optimal Trotter step count.


In [10]:
from src.metrics import (
    trotter_error_model_fit,
    predict_n_star,
    bootstrap_n_star_ci,
    bootstrap_discrete_n_star_ci,
)

fit_rows = []
n_star_rows = []

# Fit for each (processor, g, boundary)
for pid in ["A_star", "B_star"]:
    proc = PROCESSOR_REGISTRY[pid]
    for g in G_VALUES:
        sub = df_ext[(df_ext.processor == pid) & (df_ext.g == g) &
                     (df_ext.boundary == "open") & (df_ext.order == 1)].sort_values("n_steps")
        if len(sub) < 3:
            continue
        
        n_arr = sub["n_steps"].values.astype(float)
        err_nl = sub["error_noiseless"].values
        err_ny = sub["error_noisy"].values
        
        fit = trotter_error_model_fit(n_arr, err_nl, err_ny)
        n_star_meas = int(n_arr[np.argmin(err_ny)])
        ci_lo, ci_hi = bootstrap_n_star_ci(n_arr, err_ny, k_fixed=1, lambda_step=fit["Lambda"])
        ci_disc_lo, ci_disc_hi = bootstrap_discrete_n_star_ci(n_arr, err_ny)
        
        fit_rows.append({
            "processor": pid, "g": g, "boundary": "open",
            "c": round(fit["c"], 6), "k_fit": round(fit["k_fit"], 3),
            "Lambda": round(fit["Lambda"], 6),
            "R2_noiseless": round(fit["R2_noiseless"], 4),
            "R2_noisy": round(fit["R2_noisy"], 4),
            "n_star_pred": fit["n_star_pred"],
            "n_star_measured": n_star_meas,
        })
        
        n_star_rows.append({
            "processor": pid, "g": g, "n_star": n_star_meas,
            "n_star_pred": fit["n_star_pred"],
            "ci_model_lower": round(ci_lo, 1), "ci_model_upper": round(ci_hi, 1),
            "ci_disc_lower": int(ci_disc_lo), "ci_disc_upper": int(ci_disc_hi),
        })
        
        print(f"{pid} g={g}: c={fit['c']:.4f} k={fit['k_fit']:.2f} "
              f"Λ={fit['Lambda']:.5f} n*={n_star_meas} (pred={fit['n_star_pred']})")

df_fit = pd.DataFrame(fit_rows)
df_nstar = pd.DataFrame(n_star_rows)
df_fit.to_csv("results/tables/model_fit.csv", index=False)
df_nstar.to_csv("results/tables/n_star.csv", index=False)
display(df_fit)


A_star g=0.5: c=0.7498 k=1.00 Λ=0.98500 n*=4 (pred=7)


A_star g=1.0: c=0.6579 k=1.00 Λ=0.98500 n*=6 (pred=7)


A_star g=1.5: c=1.1094 k=1.00 Λ=0.98500 n*=4 (pred=9)


B_star g=0.5: c=0.7498 k=1.00 Λ=0.98500 n*=4 (pred=7)


B_star g=1.0: c=0.6579 k=1.00 Λ=0.98500 n*=6 (pred=7)


B_star g=1.5: c=1.1094 k=1.00 Λ=0.98500 n*=4 (pred=9)


,processor,g,boundary,c,k_fit,Lambda,R2_noiseless,R2_noisy,n_star_pred,n_star_measured
0,A_star,0.5,open,0.749840,1.0,0.985,0.6680,0.6140,7,4
1,A_star,1.0,open,0.657949,1.0,0.985,0.8134,0.6002,7,6
2,A_star,1.5,open,1.109407,1.0,0.985,0.6767,0.5963,9,4
3,B_star,0.5,open,0.749840,1.0,0.985,0.6680,0.6140,7,4
4,B_star,1.0,open,0.657949,1.0,0.985,0.8134,0.6002,7,6
5,B_star,1.5,open,1.109407,1.0,0.985,0.6767,0.5963,9,4


In [11]:
# ZNE and readout mitigation
mit_before = {}
mit_after  = {}

H_test = tfim_hamiltonian(4, 1.0, 1.0, "open")
proc = PROCESSOR_REGISTRY["A_star"]
sv0 = initial_state("zeros", 4)
exact_z0 = exact_trajectory(H_test, sv0, np.array([2.0]), ["Z0"])["Z0"][0]

for n_steps, label in [(2, "n=2"), (4, "n=4")]:
    qc = build_trotter_circuit(H_test, 2.0, n_steps, order=1)
    qc_t = transpile_circuit(qc, proc)
    
    # Raw (shot-based)
    shot_res = simulate_shots(qc_t, proc, ["Z0"], shots=4096)
    mit_before[label] = shot_res["Z0"]
    
    # ZNE
    try:
        zne_val = zne_extrapolate(qc_t, proc, "Z0", scale_factors=[1, 3, 5])
        mit_after[f"ZNE {label}"] = (zne_val, 0.0)
    except Exception as e:
        print(f"  ZNE failed for {label}: {e}")
        mit_after[f"ZNE {label}"] = mit_before[label]

print(f"Exact Z0 at t=2: {exact_z0:.4f}")
for k, (m, se) in {**mit_before, **mit_after}.items():
    print(f"  {k}: {m:.4f} ± {se:.4f}")

fig8_mitigation(mit_before, mit_after, exact_val=exact_z0)
plt.show()


Exact Z0 at t=2: 0.0586
  n=2: 0.4819 ± 0.0137
  n=4: 0.0781 ± 0.0156
  ZNE n=2: 0.5059 ± 0.0000
  ZNE n=4: 0.0792 ± 0.0000


## 8. Phase 7 — Figures and Report

Generating all 8 required figures.


In [12]:
import matplotlib.pyplot as plt

# Fig 2: Trajectory
g_plot = 1.0
t_arr = np.array(T_GRID)
proc_ids_plot = ["A_star"]
init_plot = "zeros"

fig2_data = {"t_grid": t_arr, "exact": EXACT[("open", g_plot, init_plot)]["Z0"],
             "proc_ids": proc_ids_plot, "n_steps_list": [1, 2, 4]}

for pid in proc_ids_plot:
    proc = PROCESSOR_REGISTRY[pid]
    H_g = tfim_hamiltonian(4, 1.0, g_plot, "open")
    sv0 = initial_state(init_plot, 4)
    for n_steps in [1, 2, 4]:
        noisy_traj = []
        for t_val in t_arr:
            qc = build_trotter_circuit(H_g, t_val if t_val > 0 else 1e-6, n_steps, order=1)
            qc_t = transpile_circuit(qc, proc)
            dm_res = simulate_density_matrix(qc_t, proc, ["Z0"])
            noisy_traj.append(dm_res["Z0"])
        fig2_data[(pid, n_steps, "noisy")] = np.array(noisy_traj)

fig2_trajectory(fig2_data)
plt.show()
print("Fig 2 saved.")


Fig 2 saved.


In [13]:
# Fig 3: U-curve (hero figure 1)
pid = "A_star"
g = 1.0
sub = df_ext[(df_ext.processor==pid) & (df_ext.g==g) & 
             (df_ext.boundary=="open") & (df_ext.order==1)].sort_values("n_steps")
n_arr  = sub["n_steps"].values.astype(float)
err_nl = sub["error_noiseless"].values
err_ny = sub["error_noisy"].values
fit = trotter_error_model_fit(n_arr, err_nl, err_ny)

fig3_u_curve(n_arr, err_ny, fit, fit["n_star_pred"], err_nl)
plt.show()
print(f"Fig 3 saved.  n*={fit['n_star_pred']} c={fit['c']:.4f} Λ={fit['Lambda']:.5f}")


Fig 3 saved.  n*=7 c=0.6579 Λ=0.98500


In [14]:
# Fig 4: n* map (hero figure 2) — noise-scale sweep
if not QUICK:
    H_test = tfim_hamiltonian(4, 1.0, 1.0, "open")
    proc = PROCESSOR_REGISTRY["A_star"]
    noise_scales = [0.25, 0.5, 1.0, 2.0, 4.0]
    t_sweep = T_GRID_RED
    
    n_star_grid = np.zeros((len(noise_scales), len(t_sweep)), dtype=int)
    for ni, ns in enumerate(noise_scales):
        for ti, t_val in enumerate(t_sweep):
            rec = recommend_steps(H_test, t_val, proc, noise_scale=ns, n_candidates=[1,2,3,4,6,8])
            n_star_grid[ni, ti] = rec["n_star"]
    
    fig4_n_star_map(n_star_grid, np.array(t_sweep), np.array(noise_scales))
    plt.show()
    print("Fig 4 saved.")
else:
    print("Fig 4 skipped in QUICK mode.")


Fig 4 saved.


In [15]:
# Fig 5: Scaling law
if not QUICK:
    sweep = noise_scale_sweep(H_test, 2.0, proc, noise_scales=[0.25, 0.5, 1.0, 2.0, 4.0])
    df_scaling = pd.DataFrame({"noise_scale": sweep["noise_scales"], "n_star": sweep["n_stars"]})
    df_scaling.to_csv("results/tables/scaling_law.csv", index=False)
    fig5_scaling_law(sweep["noise_scales"], sweep["n_stars"], sweep["fit_slope"], sweep["r_squared"])
    plt.show()
    print(f"Fig 5 saved. Fitted slope={sweep['fit_slope']:.3f} (theory order-1: -0.5)")
else:
    print("Fig 5 skipped in QUICK mode.")


Fig 5 saved. Fitted slope=-0.300 (theory order-1: -0.5)


In [16]:
# Fig 6: CX count and depth vs n, open vs ring
circ_data = {"n_steps": n_steps_ext, "proc_ids": ["A_star", "B_star"]}
for pid in ["A_star", "B_star"]:
    proc = PROCESSOR_REGISTRY[pid]
    for boundary in ["open", "ring"]:
        cx_list, depth_list = [], []
        H_g = tfim_hamiltonian(4, 1.0, 1.0, boundary)
        for n_steps in n_steps_ext:
            qc = build_trotter_circuit(H_g, 2.0, n_steps, order=1)
            qc_t = transpile_circuit(qc, proc)
            st = circuit_stats(qc_t)
            cx_list.append(st["cx"])
            depth_list.append(st["depth"])
        circ_data[(pid, boundary, "cx")] = cx_list
        circ_data[(pid, boundary, "depth")] = depth_list

fig6_cx_depth(circ_data)
plt.show()
print("Fig 6 saved.")


Fig 6 saved.


In [17]:
# Fig 7: Layout study
if not QUICK:
    H_open = tfim_hamiltonian(4, 1.0, 1.0, "open")
    proc_A = PROCESSOR_REGISTRY["A_star"]
    qc_open = build_trotter_circuit(H_open, 2.0, 2, order=1)
    
    df_layout = layout_search(qc_open, proc_A, n_spins=4, simulate_extremes=False)
    df_layout.to_csv("results/tables/layout_study.csv", index=False)
    fig7_layout(df_layout, proc_name="A_star")
    plt.show()
    print(f"Fig 7 saved. {len(df_layout)} layouts tested.")
else:
    print("Fig 7 (layout study) skipped in QUICK mode.")


Fig 7 saved. 120 layouts tested.


## 9. Phase 8 — Validation Suite

Running all 9 validation checks from README section 8.


In [18]:
print("=== Full Validation Suite ===")
val_results = run_validation(verbose=True)
n_pass = sum(val_results.values())
print(f"\n{n_pass}/{len(val_results)} checks passed.")

# Write graph metrics table
gm_rows = []
for pid, proc in PROCESSOR_REGISTRY.items():
    m = proc["graph_metrics"]
    gm_rows.append({"processor": pid, **m})
pd.DataFrame(gm_rows).to_csv("results/tables/graph_metrics.csv", index=False)
print("Wrote results/tables/graph_metrics.csv")


=== Full Validation Suite ===
  [PASS] V1_hamiltonian_2qubit
  [PASS] V2_initial_Z0
  [PASS] V3_noiseless_convergence: slope=-2.588 (expected ≈ -1)
  [PASS] V4_zero_noise_matches_noiseless: diff=0.00e+00
  [PASS] V5_swap_counts: ring_A0_swaps=2
  [PASS] V6_cx_identity: A0_cx=14, A*_cx=8, swaps=2, diff=6
  [PASS] V7_active_qubit_sim: val_4q=-0.416147, val_5q=-0.416147, diff=0.00e+00
  [PASS] V8_dm_vs_shot: DM=-0.0867, shot=-0.0991±0.0155, diff=0.0124
  [PASS] V9_n_star_accuracy: real_benchmark: measured=6, predicted=7, |gap|=1

  9/9 checks passed.

9/9 checks passed.
Wrote results/tables/graph_metrics.csv


## 10. Decision Table

`recommend_steps()` turned into a one-page lookup table.


In [19]:
from src.metrics import recommend_steps

decision_rows = []
for pid in ["A_star", "B_star"]:
    proc = PROCESSOR_REGISTRY[pid]
    for g in G_VALUES:
        for t_val in T_GRID_RED:
            H_g = tfim_hamiltonian(4, 1.0, g, "open")
            rec = recommend_steps(H_g, t_val, proc, n_candidates=N_STEPS_REQUIRED)
            decision_rows.append({
                "processor": pid, "g": g, "t": t_val,
                "n_star": rec["n_star"],
                "predicted_error": round(rec["predicted_error"], 5),
                "ci_lower": round(rec["ci_lower"], 1),
                "ci_upper": round(rec["ci_upper"], 1),
            })

df_decision = pd.DataFrame(decision_rows)
df_decision.to_csv("results/tables/decision_table.csv", index=False)
print("Decision table:")
display(df_decision)


Decision table:


,processor,g,t,n_star,predicted_error,ci_lower,ci_upper
0,A_star,0.5,0.5,1,0.03766,1.0,1.3
1,A_star,0.5,1.0,4,0.05107,2.4,3.3
2,A_star,0.5,1.5,4,0.05542,4.4,6.5
3,A_star,0.5,2.0,4,0.06344,5.9,9.8
4,A_star,1.0,0.5,2,0.03289,1.0,1.9
5,A_star,1.0,1.0,4,0.01285,3.6,5.7
6,A_star,1.0,1.5,4,0.00229,5.0,9.0
7,A_star,1.0,2.0,4,0.01492,5.6,7.8
8,A_star,1.5,0.5,4,0.01324,1.0,2.5
9,A_star,1.5,1.0,2,0.01953,3.3,6.2


---

## Summary

The optimal Trotter step count `n*` depends on:
1. **Evolution time t** — longer times need more steps (more Trotter error per step)
2. **Field strength g** — stronger field = faster dynamics = more Trotter error
3. **Noise level** — higher noise punishes more steps (more gate error)
4. **Topology** — ring Hamiltonian on poorly-matched topologies increases gate count, shifting n* lower

See `report.md` for full analysis, hypothesis outcomes, and limitations.
See `results/tables/` for all numerical results.
